# Handling Missing Values & Outliers
## Credit Card Fraud Detection Dataset

This notebook handles missing values and detects/treats outliers in the dataset.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load datasets
card_trans_df = pd.read_csv('../Datasets/card_transdata.csv')
creditcard_df = pd.read_csv('../Datasets/creditcard.csv')

print("Loaded datasets successfully")

## 1. Handle Missing Values

In [ ]:
# Check missing values percentage
def missing_values_analysis(df, name):
    missing = pd.DataFrame({
        'column': df.columns,
        'missing_count': df.isnull().sum(),
        'missing_percentage': (df.isnull().sum() / len(df)) * 100
    })
    missing = missing[missing['missing_count'] > 0].sort_values('missing_percentage', ascending=False)
    print(f"\n{name} - Missing Values Analysis:")
    print(missing)
    return missing

card_trans_missing = missing_values_analysis(card_trans_df, 'Card Transaction Data')
creditcard_missing = missing_values_analysis(creditcard_df, 'Credit Card Data')

In [ ]:
# Handle missing values based on analysis
# Options: drop rows, forward fill, backward fill, mean/median imputation, etc.
# Add your missing value handling code here

# Example: Drop rows with missing values if percentage is low
def handle_missing_values(df, threshold=5):
    # Drop columns with more than threshold% missing values
    df = df.dropna(thresh=len(df) * (100 - threshold) / 100, axis=1)
    # Drop rows with remaining missing values
    df = df.dropna()
    return df

# Apply to both datasets
card_trans_clean = handle_missing_values(card_trans_df)
creditcard_clean = handle_missing_values(creditcard_df)

print("Missing values handled successfully")

## 2. Detect & Handle Outliers

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(df, columns=None):
    if columns is None:
        columns = df.select_dtypes(include=[np.number]).columns
    
    outlier_indices = []
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        col_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].index
        outlier_indices.extend(col_outliers)
    
    return list(set(outlier_indices))

# Detect outliers
card_trans_outliers = detect_outliers_iqr(card_trans_clean)
creditcard_outliers = detect_outliers_iqr(creditcard_clean)

print(f"Card Transaction Data - Outliers detected: {len(card_trans_outliers)}")
print(f"Credit Card Data - Outliers detected: {len(creditcard_outliers)}")

In [ ]:
# Remove outliers (or you can cap them instead)
card_trans_no_outliers = card_trans_clean.drop(card_trans_outliers)
creditcard_no_outliers = creditcard_clean.drop(creditcard_outliers)

print("\nOutliers removed:")
print(f"Card Transaction Data: {card_trans_clean.shape[0]} -> {card_trans_no_outliers.shape[0]}")
print(f"Credit Card Data: {creditcard_clean.shape[0]} -> {creditcard_no_outliers.shape[0]}")

In [ ]:
# Visualize before and after
# Create box plots to show outlier removal effect
# Add your visualization code here